# URB-03 — Entregables del 18/11/2026
## Índice H-E-V y sensibilidad
Notebook complementario. **No reemplaza el piloto original.** Datos sintéticos o ilustrativos; no constituyen validación real. Ejecutar en Colab/Jupyter y descargar la carpeta de salida.


In [ ]:
import numpy as np,pandas as pd
from pathlib import Path
out=Path("salida_urb03");out.mkdir(exist_ok=True)
rng=np.random.default_rng(42);n=5
df=pd.DataFrame({"id_zona":[f"PER-{i:02d}" for i in range(1,n+1)]})
for riesgo in ["vial","inundacion","seguridad"]:
 for dim in ["H","E","V"]:df[f"{riesgo}_{dim}"]=rng.uniform(.1,.9,n).round(3)
# Entradas sintéticas normalizadas [0,1]; pesos ilustrativos, no validados.
def calcular(tab,pesos=(.4,.3,.3),metodo="aditivo"):
 salida=tab[["id_zona"]].copy()
 for riesgo in ["vial","inundacion","seguridad"]:
  x=tab[[f"{riesgo}_{d}" for d in "HEV"]].to_numpy()
  salida[f"riesgo_{riesgo}"]=x@np.array(pesos) if metodo=="aditivo" else np.prod(np.maximum(x,1e-8)**np.array(pesos),axis=1)
 salida["criticidad_total"]=salida[[c for c in salida if c.startswith("riesgo_")]].mean(axis=1)
 return salida
a=calcular(df);g=calcular(df,metodo="geometrico")
a.to_csv(out/"indice_aditivo.csv",index=False);g.to_csv(out/"indice_geometrico.csv",index=False)
filas=[]
for delta in [-.2,-.1,0,.1,.2]:
 p=np.array([.4+delta,.3,.3]);p=p/p.sum()
 z=calcular(df,p)
 filas.append({"variacion_peso_H":delta,"correlacion_ranking":a["criticidad_total"].rank().corr(z["criticidad_total"].rank(),method="spearman")})
pd.DataFrame(filas).to_csv(out/"sensibilidad.csv",index=False)
print("Cinco zonas sintéticas; dos agregaciones (aditiva y geométrica).")
print(a.round(3).to_string(index=False))